## setup BigQuery - TechZone

 este Notebook configura la conexion con Google BigQuery,
 crea el dataset y define la estructura de tablas del proyecto. 

## 1. Configuración de entorno y conexión

se cargan variables desde `.env` y se establece conexión con BigQuery.

In [ ]:
#importar librerias
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

# Cargar las variables de entorno
load_dotenv(dotenv_path="../.env")

# Obtener variables
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# configuracion de credenciales
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(Path("..") / CREDENTIALS_PATH)

# crear cliente BigQuery
client = bigquery.Client(project=PROJECT_ID)

print("Conexión lista 🚀")
print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)

Conexión lista 🚀
Proyecto: techzone-494713
Dataset: TechZone


## 2. Creación de tablas

se crean las tablas siguiendo el modelo definido por el equipo

In [ ]:
# SQL para crear tablas en BigQuery

tables_sql = [
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.customers`(
        customer_id INT64,
        first_name STRING,
        last_name STRING,
        email STRING,
        phone STRING,
        country STRING,
        city STRING,
        acquisition_channel STRING,
        registered_at TIMESTAMP
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.categories` (
        category_id INT64,
        name STRING,
        description STRING
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.products` (
        product_id INT64,
        category_id INT64,
        name STRING,
        description STRING,
        sale_price NUMERIC,
        cost_price NUMERIC,
        stock_qty INT64,
        is_active BOOL
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.orders` (
        order_id INT64,
        customer_id INT64,
        status STRING,
        shipping_address STRING,
        shipping_city STRING,
        shipping_country STRING,
        order_date TIMESTAMP,
        shipped_at TIMESTAMP,
        delivered_at TIMESTAMP
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.order_items` (
        order_item_id INT64,
        order_id INT64,
        product_id INT64,
        quantity INT64,
        purchase_unit_price NUMERIC,
        discount_amount NUMERIC
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.payments` (
        payment_id INT64,
        order_id INT64,
        payment_method STRING,
        payment_status STRING,
        amount NUMERIC,
        payment_date TIMESTAMP
    )
    ''',
    '''
    CREATE OR REPLACE TABLE `techzone-494713.TechZone.reviews` (
        review_id INT64,
        order_item_id INT64,
        rating INT64,
        comment STRING,
        review_date TIMESTAMP
    )
    '''
]

Tablas creadas Correctamente


In [ ]:
# Ejecutar creacion tablas 
for table_sql in tables_sql:
    client.query(table_sql).result()

print("Tablas creadas Correctamente")

Tablas creadas Correctamente


## 3. Validación

se verifica que las tablas fueron creadas correctamente 

In [5]:
query = """
SELECT COUNT(*) AS total_filas
FROM `techzone-494713.TechZone.customers`
"""

df = client.query(query).to_dataframe()
df

c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_filas
0,0


## 4. Preparación para carga de datos

Se define una función para cargar DataFrames en BigQuery.
Esta será usada cuando se generen los datos.

In [6]:
#Funcion para cargar DataFrame a BigQuery
#será Usada cuando tengamos datos(Hugo)
def upload_dataframe(df, table_name):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    
    job = client.load_table_from_dataframe(df, table_id)
    job.result()
    
    print(f"Datos cargados en {table_name}")